Este notebook implementa una Red Neuronal Convolucional construida desde cero para clasificar imágenes de Street View. El objetivo es superar el 24% de precisión del modelo anterior utilizando técnicas de regularización como Data Augmentation avanzado, Normalización por Lotes (BatchNormalization), y Global Average Pooling para prevenir el sobreajuste.

In [ ]:
import numpy as np
import pandas as pd
import os
import pathlib
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

## 2. Definición del Dataset
Leemos las imágenes de Kaggle o directorio local, filtramos países minoritarios y dividimos en Train/Val estricto.

In [ ]:
# Ajustar ruta según entorno (Kaggle o local)
data_dir = pathlib.Path('/kaggle/input/datasets/ubitquitin/geolocation-geoguessr-images-50k')
#current_dir = pathlib.Path(os.getcwd())
#data_dir = current_dir / 'data' / 'compressed_dataset' # Ruta local por defecto

print(f"Buscando imágenes en: {data_dir}")

filepaths = list(data_dir.glob(r'**/*.jpg'))
labels = [os.path.split(os.path.split(x)[0])[1] for x in filepaths]

df = pd.DataFrame({'Filepath': [str(x) for x in filepaths], 'Label': labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Filtrar países con pocas imágenes (Mínimo 50 para buena generalización)
MIN_IMAGENES = 50
num_fotos_por_pais = df['Label'].value_counts()
paises_validos = num_fotos_por_pais[num_fotos_por_pais >= MIN_IMAGENES].index
df_filtrado = df[df['Label'].isin(paises_validos)].copy()

print(f"Hemos mantenido {len(paises_validos)} países de {len(num_fotos_por_pais)}")
print(f"Dataset filtrado: {len(df_filtrado)} imágenes")

train_df, val_df = train_test_split(
    df_filtrado,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=df_filtrado['Label']
)

print(f"Imágenes de Entrenamiento: {len(train_df)}")
print(f"Imágenes de Validación: {len(val_df)}")

## 3. Data Generators
Aquí solo reescalamos.

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

datagen = ImageDataGenerator(rescale=1./255)

train_images = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMG_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

val_images = datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMG_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

num_classes = len(train_images.class_indices)

## 4. Construcción de la CNN Personalizada
Preprocesamiento agresivo de imágenes y regularización extensiva.

In [ ]:
def build_custom_cnn(num_classes):
    inputs = keras.Input(shape=(224, 224, 3))
    
    # --- 1. Data Augmentation Nativo Nativo agresivo ---
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.15)(x)
    x = layers.RandomZoom(0.2)(x)
    x = layers.RandomContrast(0.25)(x)
    
    # --- 2. Bloque Convolucional 1 ---
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) # 112x112
    
    # --- 3. Bloque Convolucional 2 ---
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) # 56x56
    
    # --- 4. Bloque Convolucional 3 ---
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) # 28x28
    
    # --- 5. Bloque Convolucional 4 (Extracción Profunda) ---
    x = layers.Conv2D(256, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(256, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) # 14x14
    
    # --- 6. Cabeza Clasificadora (Global Average Pooling en lugar de Flatten) ---
    # Esto reduce los parámetros inmensamente y 
    # previene el temido overfitting al forzar al modelo a promediar características espaciales.
    x = layers.GlobalAveragePooling2D()(x)
    
    x = layers.Dense(1024, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x) # 50% neuronas apagadas
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    return model

model = build_custom_cnn(num_classes)
model.summary()


## 5. Compilación y Configuración de Callbacks
Guardado inteligente del mejor modelo y control del ratio de aprendizaje (Learning Rate).

In [ ]:
opt = keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=opt,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_list = [
    # Guarda el mejor modelo basándose en val_accuracy
    callbacks.ModelCheckpoint(
        '/kaggle/working/best_custom_cnn.keras', # Ruta para Kaggle
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    # Reduce Learning Rate si validation loss se estanca en 3 épocas
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        patience=3,
        factor=0.25,
        min_lr=1e-6,
        verbose=1
    ),
    # Aborta entrenamiento tras 8 épocas de estancamiento para ahorrar recursos y evitar overfitting
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    )
]

## 6. Entrenamiento

In [ ]:
EPOCHS = 50

history = model.fit(
    train_images,
    validation_data=val_images,
    epochs=EPOCHS,
    callbacks=callbacks_list
)

## 7. Gráficas de Rendimiento

In [ ]:
plt.figure(figsize=(15, 5))

# Precisión
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Progreso de la Precisión')

# Pérdida / Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Progreso de la Pérdida (Loss)')

plt.show()

## 8. Verificación Visual de Predicciones

In [ ]:
labels_map = {v: k for k, v in train_images.class_indices.items()}

# Obtenemos lote aleatorio de validación
sample_images, sample_labels = next(val_images)
random_indices = np.random.choice(range(len(sample_images)), size=5, replace=False)

plt.figure(figsize=(20, 10))
for i, idx in enumerate(random_indices):
    image = sample_images[idx]
    true_label_idx = np.argmax(sample_labels[idx])
    true_country = labels_map[true_label_idx]
    
    prediction_scores = model.predict(np.expand_dims(image, axis=0), verbose=0)
    predicted_label_idx = np.argmax(prediction_scores)
    predicted_country = labels_map[predicted_label_idx]
    confidence = np.max(prediction_scores) * 100
    
    plt.subplot(1, 5, i + 1)
    plt.imshow(image)
    color = 'green' if true_country == predicted_country else 'red'
    plt.title(f"Real: {true_country}\nPred: {predicted_country}\nConf: {confidence:.1f}%", color=color, fontsize=12)
    plt.axis('off')

plt.tight_layout()
plt.show()